ตั้งค่าโครงสร้าง

In [2]:
!pip install -q scikit-learn pandas numpy matplotlib scipy

In [4]:
import pandas as pd
import numpy as np
import json, hashlib, os
from datetime import datetime
from scipy import stats
from scipy.special import gamma as gamma_function

# ข้อมูลไซต์
SITE_INFO = {
    "name": "GWD54",
    "station_code": "GWD54",
    "province": "ยโสธร",
    "latitude": 15.98565,
    "longitude": 104.2271899,
    "mast_height_m": 160,           # ความสูงเสา
    "data_start": "2025-10-08",
    "records_per_day": 144,         # = ราย 10 นาที (6 × 24)
}

# ผังเซนเซอร์: ชื่อคอลัมน์ ต่อด้วย ความสูง (เมตร)
SENSOR_HEIGHTS = {
    "WS160_NW": 160,   # Ch1 — ยอดเสา แขนชี้ NW
    "WS160_SE": 160,   # Ch2 — ยอดเสา แขนชี้ SE (คู่แก้เงาเสา)
    "WS140":    140,   # Ch3
    "WS120":    120,   # Ch4
    "WS100":    100,   # Ch5
    "WS80":      80,   # Ch6
    "WS60":      60,   # Ch7
}

# ทิศที่แขนยึดชี้ออก (องศา)
BOOM_BEARING_DEG = {"WS160_NW": 315.0, "WS160_SE": 135.0}

สร้างข้อมูลจำลอง

In [ ]:
def make_regional_wind(start, end, seed=42):
    """
    สร้างสัญญาณลมระดับภูมิภาครายชั่วโมง (ต้นทางร่วมของ ERA5 และไซต์)
    INPUT : start, end = ช่วงเวลา (str) | seed = ล็อกการสุ่ม
    OUTPUT: pd.Series ความเร็วลม (m/s) index = เวลารายชั่วโมง
    """
    random = np.random.default_rng(seed)
    timestamps = pd.date_range(start, end, freq="1h")

    white_noise = random.normal(0, 1, len(timestamps))
    smoothed = pd.Series(white_noise).ewm(span=24).mean().to_numpy()  # ทำให้ลมต่อเนื่อง
    smoothed = (smoothed - smoothed.mean()) / smoothed.std()

    season_factor = 1 + 0.22*np.sin(2*np.pi*(timestamps.dayofyear - 30)/365)
    year_factor_map = pd.Series(random.normal(1, 0.05, timestamps.year.nunique()),
                                index=sorted(timestamps.year.unique()))
    year_factor = timestamps.year.map(year_factor_map).to_numpy()

    wind = np.clip((6.2 + 2.3*smoothed) * season_factor * year_factor, 0.1, 30)
    return pd.Series(wind, index=timestamps)


def make_mast_data(regional_wind, start, end, seed=7):
    """
    สร้างข้อมูลเสาวัดราย 10 นาที พร้อม "ความไม่สมบูรณ์" แบบข้อมูลจริง
    INPUT : regional_wind = Series จากฟังก์ชันข้างบน | start, end = ช่วงเวลาเสา
    OUTPUT: pd.DataFrame ราย 10 นาที (144 แถว/วัน) มีคอลัมน์ตาม SENSOR_HEIGHTS
            + WD (ทิศทาง), SD, Temp, Pres, RH
    """
    random = np.random.default_rng(seed)
    timestamps = pd.date_range(start, end, freq="10min")
    n_rows = len(timestamps)

    # ดึงลมภูมิภาคมาเป็นฐาน แล้วเติมความแปรปรวนเฉพาะที่
    reference_wind = (regional_wind
                      .reindex(timestamps.union(regional_wind.index))
                      .interpolate("time").reindex(timestamps).to_numpy())
    reference_wind = np.clip(reference_wind + random.normal(0, 0.45, n_rows), 0.1, 32)

    hour_of_day = timestamps.hour + timestamps.minute/60
    day_of_year = timestamps.dayofyear

    # shear แปรตามเวลา (กลางคืนอากาศนิ่ง α สูง)
    shear_alpha = np.clip(0.20 + 0.075*np.cos(2*np.pi*(hour_of_day-3)/24)
                          + random.normal(0, 0.02, n_rows), 0.02, 0.5)

    # ดัชนีเสถียรภาพชั้นบรรยากาศ → ทำให้โปรไฟล์ "เบี่ยงจาก Power Law"
    stability = pd.Series(random.normal(0,1,n_rows)).ewm(span=18).mean().to_numpy()

    wind_direction = (58 + 30*np.sin(2*np.pi*(day_of_year-40)/365)
                      + random.normal(0, 28, n_rows)) % 360
    turbulence_frac = np.clip(0.10 + 0.05*np.exp(-reference_wind/8)
                              + random.normal(0, 0.015, n_rows), 0.03, 0.5)

    data = pd.DataFrame(index=timestamps)
    data.index.name = "timestamp"

    for sensor_name, height in SENSOR_HEIGHTS.items():
        deviation = 1 + 0.012*stability*np.log(height/100)      # ไม่ใช่ power law บริสุทธิ์
        wind = reference_wind*(height/100)**shear_alpha*deviation + random.normal(0,0.10,n_rows)
        data[sensor_name] = np.clip(wind, 0, None)

    # เงาเสาจริง: ลดความเร็ว ~6% เมื่อเซนเซอร์อยู่ท้ายลมของเสา
    for sensor_name, boom_deg in BOOM_BEARING_DEG.items():
        angle_gap = np.abs((wind_direction - boom_deg + 180) % 360 - 180)
        shadow_loss = np.clip((angle_gap - 120)/60, 0, 1) * 0.06
        data[sensor_name] = data[sensor_name] * (1 - shadow_loss)

    # ความผิดพลาดของเครื่องมือที่พบจริง
    data["WS140"] = data["WS140"] * (1 + np.linspace(0, 0.025, n_rows))  # calibration drift
    data["WS80"]  = data["WS80"] + 0.08                                   # offset ค้าง

    data["WD"]   = wind_direction
    data["SD"]   = np.clip(data["WS160_NW"]*turbulence_frac, 0.01, None)
    data["Temp"] = (27 + 4.5*np.sin(2*np.pi*(hour_of_day-9)/24)
                    + 3*np.sin(2*np.pi*(day_of_year-100)/365) + random.normal(0,0.4,n_rows))
    data["Pres"] = 1009 + 3*np.sin(2*np.pi*(day_of_year-15)/365) + random.normal(0,1.5,n_rows)
    data["RH"]   = np.clip(72 - 16*np.sin(2*np.pi*(hour_of_day-9)/24)
                           + random.normal(0,4,n_rows), 20, 100)

    # spike ผิดปกติ
    spike_mask = random.random(n_rows) < 0.0008
    data.loc[spike_mask,"WS100"] *= random.uniform(1.8, 3.0, spike_mask.sum())

    # เซนเซอร์ค้าง 6 ครั้ง
    for _ in range(6):
        start_i = random.integers(0, n_rows-40)
        length  = random.integers(8, 40)
        data.iloc[start_i:start_i+length, data.columns.get_loc("WS120")] = data["WS120"].iloc[start_i]

    # ข้อมูลหาย: กระจาย 12% + logger ดับ 2 ช่วง
    missing = random.random(n_rows) < 0.12
    for outage_start, n_days in [("2026-01-15", 5), ("2026-04-02", 9)]:
        i = timestamps.get_indexer([pd.Timestamp(outage_start)], method="nearest")[0]
        missing[i : i + n_days*144] = True
    data.loc[missing, :] = np.nan

    return data


# เรียกใช้
regional_wind = make_regional_wind("2006-01-01", "2026-10-07 23:00")

era5_reference = pd.DataFrame(
    {"era_ws": np.clip(0.82*regional_wind.to_numpy()
                       + np.random.default_rng(3).normal(0,0.9,len(regional_wind)) - 0.3,
                       0.1, 30)},
    index=regional_wind.index)

raw_data = make_mast_data(regional_wind, "2025-10-08", "2026-10-07 23:50")

n_days = (raw_data.index[-1] - raw_data.index[0]).days + 1
print("ข้อมูลดิบ:", raw_data.shape)
print("แถวต่อวัน = %.0f  (ควรได้ 144)" % (len(raw_data)/n_days))
print("Coverage  = %.2f%%" % (raw_data["WS100"].notna().mean()*100))

Data Versioning

In [ ]:
VERSION_DIR = "data_versions"

def save_data_version(data, note=""):
    """
    บันทึก snapshot ข้อมูล + metadata เพื่อให้ผลการเทรนย้อนกลับมาทำซ้ำได้
    INPUT : data = DataFrame ที่จะ freeze | note = คำอธิบายสั้น ๆ
    OUTPUT: version_id (str) เช่น 'v20260803_102311_a3ad7d0c'
    """
    os.makedirs(VERSION_DIR, exist_ok=True)

    # hash เนื้อข้อมูล → ถ้าข้อมูลเหมือนเดิม hash เหมือนเดิม (ตรวจได้ว่าเปลี่ยนจริงไหม)
    content_hash = hashlib.md5(
        pd.util.hash_pandas_object(data.fillna(-999)).values.tobytes()
    ).hexdigest()[:8]

    version_id = f"v{datetime.now():%Y%m%d_%H%M%S}_{content_hash}"
    folder = os.path.join(VERSION_DIR, version_id)
    os.makedirs(folder, exist_ok=True)

    data.to_csv(os.path.join(folder, "data.csv.gz"), compression="gzip")

    metadata = {
        "version_id": version_id,
        "content_hash": content_hash,
        "created_at": datetime.now().isoformat(),
        "n_rows": len(data),
        "columns": list(data.columns),
        "period": [str(data.index.min()), str(data.index.max())],
        "coverage_pct": {c: round(float(data[c].notna().mean()*100), 2) for c in data.columns},
        "note": note,
        "site": SITE_INFO,
    }
    json.dump(metadata, open(os.path.join(folder, "metadata.json"), "w"),
              indent=2, ensure_ascii=False)
    return version_id


def load_data_version(version_id):
    """
    INPUT : version_id (str)
    OUTPUT: (DataFrame, metadata dict)
    """
    folder = os.path.join(VERSION_DIR, version_id)
    data = pd.read_csv(os.path.join(folder, "data.csv.gz"), index_col=0, parse_dates=True)
    metadata = json.load(open(os.path.join(folder, "metadata.json")))
    return data, metadata


def list_data_versions():
    """OUTPUT: DataFrame สรุปทุกเวอร์ชันที่มี"""
    if not os.path.isdir(VERSION_DIR):
        return pd.DataFrame()
    rows = []
    for v in sorted(os.listdir(VERSION_DIR)):
        meta_path = os.path.join(VERSION_DIR, v, "metadata.json")
        if not os.path.exists(meta_path):
            continue
        m = json.load(open(meta_path))
        rows.append({"version_id": m["version_id"], "created": m["created_at"][:19],
                     "rows": m["n_rows"], "period_end": m["period"][1][:10],
                     "note": m["note"]})
    return pd.DataFrame(rows)


# ── บันทึกและโหลดกลับ (ทุกการเทรนต้องอ้างอิง version_id เสมอ) ──
DATA_VERSION = save_data_version(raw_data, "snapshot ทดลอง")
print("บันทึกเวอร์ชัน:", DATA_VERSION)

raw_data, data_metadata = load_data_version(DATA_VERSION)
print(list_data_versions().to_string(index=False))

QC

In [ ]:
def qc_wind_speed(series, v_min=0.0, v_max=40.0, stuck_steps=6, spike_ratio=2.0):
    """
    กรองค่าที่น่าสงสัยออกจากคอลัมน์ความเร็วลม
    INPUT : series      = Series ความเร็วลมราย 10 นาที
            v_min/v_max = ช่วงค่าที่ยอมรับ (m/s)
            stuck_steps = ค่าซ้ำติดกันกี่ช่วงถือว่าเซนเซอร์ค้าง (6 ช่วง × 10 นาที = 1 ชม.)
            spike_ratio = สูงกว่ามัธยฐานรอบข้างกี่เท่าถือว่า spike
    OUTPUT: Series เดิม แต่ค่าน่าสงสัยกลายเป็น NaN
    """
    cleaned = series.where((series >= v_min) & (series <= v_max))

    # ค่าค้าง: นับว่าค่าเดิมซ้ำติดกันกี่ช่วง
    is_same = cleaned.diff().abs() < 1e-6
    run_length = is_same.groupby((~is_same).cumsum()).cumcount() + 1
    cleaned = cleaned.mask(run_length >= stuck_steps)

    # spike: สูงกว่ามัธยฐาน 3 ชม.รอบข้างมากผิดปกติ
    rolling_median = cleaned.rolling(18, center=True, min_periods=6).median()
    cleaned = cleaned.mask(cleaned > rolling_median*spike_ratio + 2.0)
    return cleaned


qc_data = raw_data.copy()
for sensor_name in SENSOR_HEIGHTS:
    qc_data[sensor_name] = qc_wind_speed(qc_data[sensor_name])

print("Coverage หลัง QC (%):")
print((qc_data[list(SENSOR_HEIGHTS)].notna().mean()*100).round(2).to_string())

# ── TI คำนวณตรง ๆ ได้เลย เพราะ logger ให้ SD ของช่วง 10 นาทีมาแล้ว ──
qc_data["TI"] = qc_data["SD"] / qc_data["WS160_NW"]
bin_15ms = (qc_data["WS160_NW"] >= 14) & (qc_data["WS160_NW"] < 16)
print("\nTI ที่ช่วง 15 m/s (ใช้เลือกคลาสกังหัน ตาม IEC 61400-1) = %.4f"
      % qc_data.loc[bin_15ms, "TI"].mean())

Tower Shadow

In [ ]:
def angle_gap(angle_a, angle_b):
    """ผลต่างเชิงมุม 0-180° (จัดการการวน 0/360 ให้แล้ว)"""
    return np.abs((angle_a - angle_b + 180) % 360 - 180)

gap_to_nw = angle_gap(qc_data["WD"], BOOM_BEARING_DEG["WS160_NW"])
gap_to_se = angle_gap(qc_data["WD"], BOOM_BEARING_DEG["WS160_SE"])

# เลือกเซนเซอร์ที่ "แขนชี้เข้าหาลม" = อยู่เหนือลมของเสา = ไม่โดนบัง
use_nw_sensor = gap_to_nw <= gap_to_se
qc_data["WS160"] = pd.Series(
    np.where(use_nw_sensor, qc_data["WS160_NW"], qc_data["WS160_SE"]),
    index=qc_data.index)
qc_data["WS160"] = qc_data["WS160"].fillna(qc_data["WS160_NW"].fillna(qc_data["WS160_SE"]))

print("WS160 coverage = %.2f%%" % (qc_data["WS160"].notna().mean()*100))
print("สัดส่วนที่เลือกใช้ตัว NW = %.1f%%" % (use_nw_sensor.mean()*100))

สร้าง Feature + หา α

In [ ]:
def build_features(data):
    """
    เพิ่มคอลัมน์ feature ที่โมเดลต้องใช้
    INPUT : DataFrame ที่ผ่าน QC แล้ว (ต้องมี WD, WS60, WS100)
    OUTPUT: DataFrame เดิม + คอลัมน์ใหม่ 7 ตัว
    """
    result = data.copy()

    # ทิศทางลม → sin/cos (เพราะ 0° กับ 360° คือทิศเดียวกัน)
    direction_rad = np.deg2rad(result["WD"])
    result["wd_sin"] = np.sin(direction_rad)
    result["wd_cos"] = np.cos(direction_rad)

    # เวลาในวัน → sin/cos (23:00 กับ 01:00 ควรใกล้กัน)
    hour_of_day = result.index.hour + result.index.minute/60
    result["hour_sin"] = np.sin(2*np.pi*hour_of_day/24)
    result["hour_cos"] = np.cos(2*np.pi*hour_of_day/24)

    # ฤดูกาล → sin/cos (31 ธ.ค. กับ 1 ม.ค. ควรใกล้กัน)
    day_of_year = result.index.dayofyear
    result["season_sin"] = np.sin(2*np.pi*day_of_year/365)
    result["season_cos"] = np.cos(2*np.pi*day_of_year/365)

    # ผลต่างลมระหว่างชั้นล่าง — บอกใบ้สภาพชั้นบรรยากาศ
    result["shear_low"] = result["WS100"] - result["WS60"]
    return result


featured_data = build_features(qc_data)

FEATURE_COLUMNS = ["WS60", "WS80", "WS100", "shear_low",
                   "wd_sin", "wd_cos", "Temp", "Pres", "RH",
                   "hour_sin", "hour_cos", "season_sin", "season_cos"]
TARGET_COLUMN = "WS160"

# ── α จริงของไซต์ (แทนที่ 0.185 ที่ไม่มีที่มา) ──
valid_for_alpha = (featured_data["WS100"] > 3) & (featured_data["WS160"] > 3)
featured_data["alpha_observed"] = np.nan
featured_data.loc[valid_for_alpha, "alpha_observed"] = (
    np.log(featured_data.loc[valid_for_alpha,"WS160"] / featured_data.loc[valid_for_alpha,"WS100"])
    / np.log(160/100))

SITE_ALPHA = featured_data["alpha_observed"].median()
print("α ของไซต์ (median) = %.4f" % SITE_ALPHA)
print("ช่วง 25-75%%        = %.4f – %.4f"
      % (featured_data["alpha_observed"].quantile(.25),
         featured_data["alpha_observed"].quantile(.75)))

# ── แบ่ง train/test ตามเวลา (ห้ามสุ่ม) ──
model_data = featured_data[FEATURE_COLUMNS + [TARGET_COLUMN, "alpha_observed"]].dropna()
split_time = model_data.index[int(len(model_data)*0.7)]
train_set = model_data.loc[:split_time]
test_set  = model_data.loc[split_time:]

print("\nใช้ได้ %d แถว | สอน %d | สอบ %d" % (len(model_data), len(train_set), len(test_set)))
print("ช่วงสอน:", train_set.index.min().date(), "→", train_set.index.max().date())
print("ช่วงสอบ:", test_set.index.min().date(),  "→", test_set.index.max().date())

เทรนและวัดผล

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_scores = []

def evaluate(y_true, y_pred, label):
    """
    คำนวณ metric และเก็บลง model_scores
    INPUT : y_true = ค่าจริง | y_pred = ค่าที่ทำนาย | label = ชื่อโมเดล
    OUTPUT: dict ผล metric (และพิมพ์ออกจอ)
    """
    above_3ms = y_true > 3      # ตัดลมอ่อนออกตอนคิด MAPE
    score = {
        "model": label,
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE": float(np.mean(np.abs((y_true[above_3ms]-y_pred[above_3ms])
                                     / y_true[above_3ms]))*100),
        "R2":   r2_score(y_true, y_pred),
    }
    model_scores.append(score)
    print("  %-22s MAE=%.3f  RMSE=%.3f  MAPE=%.2f%%  R²=%.4f"
          % (label, score["MAE"], score["RMSE"], score["MAPE"], score["R2"]))
    return score


X_train, y_train = train_set[FEATURE_COLUMNS], train_set[TARGET_COLUMN]
X_test,  y_test  = test_set[FEATURE_COLUMNS],  test_set[TARGET_COLUMN]

print("ผลการทดสอบ:")
# ── Baseline ต้องมาก่อนเสมอ ──
evaluate(y_test, X_test["WS100"]*(160/100)**SITE_ALPHA, "Power Law (baseline)")
evaluate(y_test, LinearRegression().fit(X_train, y_train).predict(X_test), "Linear Regression")

# ── ML ──
model_rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                 max_features="sqrt", random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)
evaluate(y_test, model_rf.predict(X_test), "Random Forest")

# ── RF ที่ทำนาย α (ใช้ตอนต้องขยายเกินเซนเซอร์บนสุด) ──
model_rf_alpha = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                       max_features="sqrt", random_state=42, n_jobs=-1)
model_rf_alpha.fit(X_train, train_set["alpha_observed"])
evaluate(y_test, X_test["WS100"]*(160/100)**model_rf_alpha.predict(X_test), "RF ผ่าน α")

ทำนายที่ Hub Height + MCP

In [ ]:
HUB_HEIGHT_M = 150      # 🔧 สมมติ — ต้องเปลี่ยนเป็นค่าจริงของกังหันที่เลือก

all_features = featured_data[FEATURE_COLUMNS].dropna()
predicted_alpha = model_rf_alpha.predict(all_features)
wind_at_hub = pd.Series(
    all_features["WS100"].to_numpy() * (HUB_HEIGHT_M/100)**predicted_alpha,
    index=all_features.index, name="wind_at_hub")
print("ลมที่ hub %d ม.: เฉลี่ย %.3f m/s" % (HUB_HEIGHT_M, wind_at_hub.mean()))


def add_mcp_features(data):
    """เพิ่ม feature ฤดู/ชั่วโมง สำหรับโมเดล MCP"""
    result = data.copy()
    result["month_sin"] = np.sin(2*np.pi*result.index.dayofyear/365)
    result["month_cos"] = np.cos(2*np.pi*result.index.dayofyear/365)
    result["hour_sin"]  = np.sin(2*np.pi*result.index.hour/24)
    result["hour_cos"]  = np.cos(2*np.pi*result.index.hour/24)
    return result

site_hourly = wind_at_hub.resample("1h").mean().dropna()
overlap = add_mcp_features(era5_reference.join(site_hourly.rename("site"),
                                               how="inner").dropna())

MCP_FEATURES = ["era_ws", "month_sin", "month_cos", "hour_sin", "hour_cos"]
mcp_split = overlap.index[int(len(overlap)*0.7)]

model_mcp = RandomForestRegressor(n_estimators=300, min_samples_leaf=10,
                                  random_state=42, n_jobs=-1)
model_mcp.fit(overlap.loc[:mcp_split, MCP_FEATURES], overlap.loc[:mcp_split, "site"])

print("\nช่วงทับซ้อน %d ชั่วโมง" % len(overlap))
evaluate(overlap.loc[mcp_split:, "site"],
         model_mcp.predict(overlap.loc[mcp_split:, MCP_FEATURES]), "MCP (ERA5→ไซต์)")

# ── ขยายย้อนหลัง 20 ปี ──
longterm = add_mcp_features(era5_reference)
longterm["wind_longterm"] = model_mcp.predict(longterm[MCP_FEATURES])

annual_mean = longterm["wind_longterm"].resample("YS").mean()
INTERANNUAL_VARIABILITY = float(annual_mean.std()/annual_mean.mean())

print("ลมระยะยาว เฉลี่ย %.3f m/s | IAV = %.2f%%"
      % (longterm["wind_longterm"].mean(), INTERANNUAL_VARIABILITY*100))
print("ปีที่วัดจริงต่างจากค่าระยะยาว %+.2f%%"
      % ((wind_at_hub.mean()/longterm["wind_longterm"].mean()-1)*100))

Weibull → AEP → P50/P90

In [ ]:
long_term_wind = longterm["wind_longterm"].dropna().to_numpy()
long_term_wind = long_term_wind[long_term_wind > 0]

weibull_k, _, weibull_A = stats.weibull_min.fit(long_term_wind, floc=0)
print("Weibull: k=%.3f  A=%.3f m/s  (mean=%.3f)"
      % (weibull_k, weibull_A, weibull_A*gamma_function(1+1/weibull_k)))

# ── กังหันสมมติ 🔧 ต้องเปลี่ยนเป็นรุ่นจริง ──
TURBINE = {"model":"สมมติ 4.5 MW", "rated_kw":4500.0,
           "cut_in_ms":3.0, "rated_ms":12.0, "cut_out_ms":25.0, "n_turbines":10}

def power_curve(wind_speed, turbine=TURBINE):
    """
    INPUT : wind_speed = array ความเร็วลม (m/s)
    OUTPUT: array กำลังไฟ (kW)
    """
    power = np.zeros_like(wind_speed, dtype=float)
    ramp = (wind_speed >= turbine["cut_in_ms"]) & (wind_speed < turbine["rated_ms"])
    power[ramp] = turbine["rated_kw"] * (
        (wind_speed[ramp]**3 - turbine["cut_in_ms"]**3)
        / (turbine["rated_ms"]**3 - turbine["cut_in_ms"]**3))
    power[(wind_speed >= turbine["rated_ms"]) & (wind_speed <= turbine["cut_out_ms"])] = turbine["rated_kw"]
    return power

# ── Air Density: ρ = P/(R·T) ──
temp_kelvin = qc_data["Temp"].mean() + 273.15
pressure_pa = qc_data["Pres"].mean() * 100
AIR_DENSITY = pressure_pa / (287.05 * temp_kelvin)

# ── AEP = Σ [P(v) × f(v) × 8760] ──
bin_edges   = np.arange(0, 30.5, 0.5)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_freq    = np.diff(stats.weibull_min.cdf(bin_edges, weibull_k, 0, weibull_A))
bin_freq   /= bin_freq.sum()

wind_corrected = bin_centers * (AIR_DENSITY/1.225)**(1/3)   # ปรับตาม air density
aep_gross_per_turbine = np.sum(power_curve(wind_corrected)*bin_freq*8760)/1e6

LOSSES = {"wake":0.05, "availability":0.03, "electrical":0.02, "other":0.02}  # 🔧 สมมติ
aep_net_per_turbine = aep_gross_per_turbine
for loss in LOSSES.values():
    aep_net_per_turbine *= (1 - loss)

AEP_NET_FARM = aep_net_per_turbine * TURBINE["n_turbines"]

print("ρ = %.4f kg/m³ | AEP/ตัว: gross=%.3f net=%.3f GWh"
      % (AIR_DENSITY, aep_gross_per_turbine, aep_net_per_turbine))

# ── P50 / P75 / P90 ──
UNCERTAINTY = {                       # 🔧 ค่าสมมติทั้งหมด
    "การวัด":            0.030,
    "ขยายความสูง":       0.040,
    "MCP / ระยะยาว":     0.035,
    "Power curve":       0.050,
    "Wake / ความพร้อม":   0.030,
    "ความผันผวนรายปี":    INTERANNUAL_VARIABILITY,
}
SIGMA_TOTAL = float(np.sqrt(sum(u**2 for u in UNCERTAINTY.values())))

P50 = AEP_NET_FARM
P75 = AEP_NET_FARM * (1 - stats.norm.ppf(0.75)*SIGMA_TOTAL)
P90 = AEP_NET_FARM * (1 - stats.norm.ppf(0.90)*SIGMA_TOTAL)
CAPACITY_FACTOR = AEP_NET_FARM*1e6 / (TURBINE["rated_kw"]*TURBINE["n_turbines"]*8760) * 100

วิเคราะห์ความคุ้มทุน

In [ ]:
ECONOMICS = {                          # 🔧 ค่าสมมติทั้งหมด ต้องใช้ตัวเลขโครงการจริง
    "capex_musd_per_mw":     1.30,
    "opex_musd_per_mw_year": 0.045,
    "ppa_usd_per_mwh":       85.0,
    "project_life_years":    20,
    "discount_rate":         0.08,
    "annual_degradation":    0.005,
}
capacity_mw = TURBINE["rated_kw"]*TURBINE["n_turbines"]/1000
CAPEX_MUSD  = ECONOMICS["capex_musd_per_mw"] * capacity_mw

def financial_analysis(aep_gwh):
    """
    INPUT : aep_gwh = พลังงานต่อปี (GWh)
    OUTPUT: dict {NPV (MUSD), IRR (%), Payback (ปี), LCOE (USD/MWh)}
    """
    cash_flow = [-CAPEX_MUSD]
    for year in range(1, ECONOMICS["project_life_years"]+1):
        energy_mwh = aep_gwh*1000 * (1-ECONOMICS["annual_degradation"])**(year-1)
        revenue = energy_mwh * ECONOMICS["ppa_usd_per_mwh"] / 1e6
        opex    = ECONOMICS["opex_musd_per_mw_year"] * capacity_mw
        cash_flow.append(revenue - opex)
    cash_flow = np.array(cash_flow)

    npv = sum(c/(1+ECONOMICS["discount_rate"])**i for i, c in enumerate(cash_flow))
    roots = np.roots(cash_flow[::-1])
    roots = roots[np.isreal(roots) & (roots.real > 0)].real
    irr = (1/roots.max()-1)*100 if len(roots) else float("nan")
    cumulative = np.cumsum(cash_flow)
    payback = int(np.argmax(cumulative > 0)) if (cumulative > 0).any() else None

    discounted_cost = CAPEX_MUSD + sum(
        ECONOMICS["opex_musd_per_mw_year"]*capacity_mw/(1+ECONOMICS["discount_rate"])**y
        for y in range(1, ECONOMICS["project_life_years"]+1))
    discounted_energy = sum(
        aep_gwh*1000*(1-ECONOMICS["annual_degradation"])**(y-1)/(1+ECONOMICS["discount_rate"])**y
        for y in range(1, ECONOMICS["project_life_years"]+1))
    lcoe = discounted_cost/discounted_energy*1e6

    return {"NPV": npv, "IRR": irr, "Payback": payback, "LCOE": lcoe}

รายงานสรุป

In [ ]:
print("="*78)
print("  รายงานสรุปผลการประเมินศักยภาพลม (Wind Resource Assessment)")
print("="*78)
print(f"  ไซต์          : {SITE_INFO['name']} (รหัส {SITE_INFO['station_code']}) "
      f"จ.{SITE_INFO['province']}")
print(f"  พิกัด          : {SITE_INFO['latitude']}, {SITE_INFO['longitude']}")
print(f"  ความสูงเสา     : {SITE_INFO['mast_height_m']} ม.")
print(f"  ช่วงข้อมูลที่วัด : {raw_data.index.min():%d %b %Y} – {raw_data.index.max():%d %b %Y}")
print(f"  ข้อมูลระยะยาว   : {longterm.index.min():%Y} – {longterm.index.max():%Y} (ERA5)")
print(f"  Data Version  : {DATA_VERSION}")
print(f"  วันที่ออกรายงาน : {datetime.now():%d %b %Y %H:%M}")
print("-"*78)
print("  [ ลักษณะลม ]")
print(f"  Data coverage        : {raw_data['WS100'].notna().mean()*100:.2f}%  "
      f"{'✓' if raw_data['WS100'].notna().mean()>0.95 else '⚠ ต่ำกว่าเกณฑ์ 95%'}")
print(f"  Wind shear (α)       : {SITE_ALPHA:.4f}")
print(f"  TI @ 15 m/s          : {qc_data.loc[bin_15ms,'TI'].mean():.4f}")
print(f"  Weibull k / A        : {weibull_k:.3f} / {weibull_A:.3f} m/s")
print(f"  ลมที่ hub {HUB_HEIGHT_M} ม.     : {wind_at_hub.mean():.3f} m/s (ปีที่วัด)")
print(f"  ลมระยะยาว 20 ปี       : {longterm['wind_longterm'].mean():.3f} m/s")
print(f"  ความผันผวนรายปี (IAV) : {INTERANNUAL_VARIABILITY*100:.2f}%")
print("-"*78)
print("  [ ผลผลิตพลังงาน ]")
print(f"  กังหัน               : {TURBINE['model']} × {TURBINE['n_turbines']} ตัว "
      f"({capacity_mw:.0f} MW)")
print(f"  Air density          : {AIR_DENSITY:.4f} kg/m³")
print(f"  Capacity Factor      : {CAPACITY_FACTOR:.2f}%")
print(f"  ความไม่แน่นอนรวม (σ)  : {SIGMA_TOTAL*100:.2f}%")
print(f"  P50 / P75 / P90      : {P50:.2f} / {P75:.2f} / {P90:.2f} GWh/ปี")
print("-"*78)
print("  [ ความคุ้มค่าการลงทุน ]  CAPEX = %.1f MUSD | PPA = %.0f USD/MWh | อายุ %d ปี"
      % (CAPEX_MUSD, ECONOMICS["ppa_usd_per_mwh"], ECONOMICS["project_life_years"]))
print(f"  {'กรณี':<6}{'AEP(GWh)':>10}{'NPV(MUSD)':>12}{'IRR(%)':>9}"
      f"{'คืนทุน(ปี)':>12}{'LCOE(USD/MWh)':>16}")
for label, aep in [("P50", P50), ("P75", P75), ("P90", P90)]:
    r = financial_analysis(aep)
    print(f"  {label:<6}{aep:>10.2f}{r['NPV']:>12.2f}{r['IRR']:>9.2f}"
          f"{str(r['Payback']):>12}{r['LCOE']:>16.2f}")
print("-"*78)
p90_result = financial_analysis(P90)
verdict = "✓ คุ้มค่า" if p90_result["NPV"] > 0 else "✗ ยังไม่คุ้มค่า"
print(f"  ข้อสรุป (เกณฑ์ธนาคารใช้ P90) : {verdict}  "
      f"(NPV = {p90_result['NPV']:.2f} MUSD, IRR = {p90_result['IRR']:.2f}%)")
print("="*78)
print("  ⚠️ ตัวเลขทั้งหมดมาจากข้อมูลจำลอง ห้ามนำไปใช้อ้างอิง")